# Produce and consume over REST

This notebook produces records to a topic and reads them back over the Diaspora Event Fabric's web service, with plain HTTP requests. Both run over HTTPS on port 443 with your Globus login, so they need no Kafka client and no access key.

- The produce routes take the request bodies of [Confluent's REST API v3](https://docs.confluent.io/platform/current/kafka-rest/api.html#records-v3), under `/api/v3/{namespace}/topics/{topic}`.
- The consumer routes take the paths and bodies of [Confluent REST Proxy v2](https://docs.confluent.io/platform/current/kafka-rest/api.html#consumers-v2), under `/api/v2/consumers`.

The [service's API reference](https://github.com/haochenpan/diaspora-service/blob/main/docs/web/api.md) gives every route, limit and error code. The last section does the same steps with the SDK's `Client.produce` and `RESTConsumer`. For login options and the Kafka clients, see [DiasporaDemo.ipynb](DiasporaDemo.ipynb).

The notebook needs `diaspora-event-sdk` 0.5.0 or later and `requests`:

```bash
pip install "diaspora-event-sdk>=0.5" requests
```

In [ ]:
import base64
import json
import time
import uuid

import requests

from diaspora_event_sdk import Client, RESTConsumer

c = Client()  # opens a Globus login the first time
base_url = c.web_service_address.rstrip("/")
namespace = c.namespace
# A login through a GlobusApp keeps its authorizer in the app; a client
# built with an authorizer keeps it in the web client.
authorizer = c.web_client.authorizer or c.app.get_authorizer(c.DIASPORA_RESOURCE_SERVER)


def auth_headers():
    # The authorizer refreshes the access token when it expires
    return {
        "Subject": c.subject_openid,
        "Authorization": authorizer.get_authorization_header(),
    }


print(f"Service:   {base_url}")
print(f"Namespace: {namespace}")

## Create a topic

Topics are created with the SDK's management call. Each run of this notebook uses a new topic and a new consumer group, and deletes the topic at the end.

In [ ]:
run = uuid.uuid4().hex[:8]
topic = f"rest-demo-{run}"
group = f"rest-demo-{run}"
print(json.dumps(c.create_topic(topic), indent=2))
time.sleep(3)  # a new topic can take a moment to reach every broker

## Produce records

`POST /api/v3/{namespace}/topics/{topic}/records` writes one record and replies once every in-sync replica has it. A key or value is `{"type", "data"}`, where the type is `JSON` (any JSON value), `STRING` (text) or `BINARY` (base64). Header values are base64. The body is sent as `application/json`.

In [ ]:
records_url = f"{base_url}/api/v3/{namespace}/topics/{topic}/records"
resp = requests.post(
    records_url,
    headers=auth_headers(),
    json={
        "key": {"type": "STRING", "data": "job-42"},
        "value": {"type": "JSON", "data": {"step": 1, "loss": 0.52}},
        "headers": [{"name": "trace", "value": base64.b64encode(b"\x00\x01").decode()}],
    },
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

`records:batch` writes up to 500 records, each with an `id` that is distinct in the batch. The service answers 207 and lists each entry under `successes` or `failures`, so one bad entry does not stop the others. Here the last entry's timestamp has no UTC offset, so that entry fails with error code 40005.

In [ ]:
resp = requests.post(
    f"{records_url}:batch",
    headers=auth_headers(),
    json={
        "entries": [
            {"id": "2", "value": {"type": "JSON", "data": {"step": 2, "loss": 0.41}}},
            {"id": "3", "value": {"type": "JSON", "data": {"step": 3, "loss": 0.35}}},
            {"id": "4", "value": {"type": "STRING", "data": "done"}},
            {
                "id": "bad",
                "value": {"type": "JSON", "data": {"step": 5}},
                "timestamp": "2026-10-26T15:04:05",
            },
        ]
    },
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## Create a consumer instance

A consumer instance belongs to a consumer group and reads in your default namespace: topics are named without the namespace, and the group `rest-demo-…` is the Kafka consumer group `{namespace}.rest-demo-…`. The `json` format returns JSON keys and values as JSON and text as strings; the `binary` format returns every key and value in base64.

The consumer routes take request bodies as `application/vnd.kafka.v2+json`. The reply's `base_uri` is the instance's absolute URL, and every later request goes to it.

In [ ]:
V2 = "application/vnd.kafka.v2+json"
resp = requests.post(
    f"{base_url}/api/v2/consumers/{group}",
    headers={**auth_headers(), "Content-Type": V2},
    json={"name": "demo-inst", "format": "json", "auto.offset.reset": "earliest"},
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))
instance = resp.json()["base_uri"]

## Subscribe and fetch

A subscribe answers 204. A fetch from a `json` instance must send `Accept: application/vnd.kafka.json.v2+json`; without it, the service refuses the fetch with 406. The group assigns the topic's partition during the first fetch, so that fetch can return no records.

In [ ]:
def v2_headers():
    return {**auth_headers(), "Content-Type": V2}


ACCEPT_JSON = {"Accept": "application/vnd.kafka.json.v2+json"}

resp = requests.post(
    f"{instance}/subscription", headers=v2_headers(), json={"topics": [topic]}
)
print("subscribe:", resp.status_code)
print(
    "subscription:",
    requests.get(f"{instance}/subscription", headers=v2_headers()).json(),
)


def fetch(timeout_ms=5000):
    resp = requests.get(
        f"{instance}/records",
        headers={**v2_headers(), **ACCEPT_JSON},
        params={"timeout": timeout_ms},
    )
    resp.raise_for_status()
    return resp.json()


records = []
for _ in range(5):
    records += fetch()
    if len(records) >= 4:
        break
for record in records:
    print(record["offset"], record["key"], record["value"], record["headers"])

## Commit offsets

A commit without a body commits the position of each assigned partition: the record after the last one fetched. It answers 200 with `[]`. `GET …/offsets` takes the partitions in its body and returns the group's committed offset for each, which is the next record to read.

In [ ]:
resp = requests.post(f"{instance}/offsets", headers=v2_headers())
print("commit:", resp.status_code, resp.json())

partitions = {"partitions": [{"topic": topic, "partition": 0}]}
resp = requests.get(f"{instance}/offsets", headers=v2_headers(), json=partitions)
print("committed:", json.dumps(resp.json(), indent=2))

A commit can also name offsets: for each partition, the offset of the last record processed. The group then commits the one after it.

In [ ]:
offsets = {"offsets": [{"topic": topic, "partition": 0, "offset": 1}]}
print(
    "commit:",
    requests.post(f"{instance}/offsets", headers=v2_headers(), json=offsets).json(),
)
resp = requests.get(f"{instance}/offsets", headers=v2_headers(), json=partitions)
print("committed:", resp.json()["offsets"])

## Seek

The three seeks move the instance's position in assigned partitions; each answers 204. Every topic has one partition, 0.

In [ ]:
resp = requests.post(
    f"{instance}/positions/beginning", headers=v2_headers(), json=partitions
)
print("seek to beginning:", resp.status_code, "->", [r["offset"] for r in fetch()])

offsets = {"offsets": [{"topic": topic, "partition": 0, "offset": 2}]}
resp = requests.post(f"{instance}/positions", headers=v2_headers(), json=offsets)
print("seek to offset 2:", resp.status_code, "->", [r["offset"] for r in fetch()])

resp = requests.post(f"{instance}/positions/end", headers=v2_headers(), json=partitions)
print("seek to end:", resp.status_code, "->", fetch(timeout_ms=2000))

## Assign a partition

In place of a subscription, an instance can be assigned partitions, and the group then does not assign them. An instance does one or the other, so it unsubscribes first.

In [ ]:
print(
    "unsubscribe:",
    requests.delete(f"{instance}/subscription", headers=v2_headers()).status_code,
)
resp = requests.post(f"{instance}/assignments", headers=v2_headers(), json=partitions)
print("assign:", resp.status_code)
print(
    "assignments:", requests.get(f"{instance}/assignments", headers=v2_headers()).json()
)

requests.post(f"{instance}/positions/beginning", headers=v2_headers(), json=partitions)
print("fetch:", [(r["offset"], r["value"]) for r in fetch()])

## Delete the instance

Delete an instance when you are done with it: a user may hold 4 instances at once. The service also drops every instance when it is deployed or restarted, and closes one that is idle for 5 minutes. A request to an instance that is gone gets 404 with error code 40403, as this one does after the delete. A client then creates the instance again, subscribes or assigns again, and continues from the group's committed offsets.

In [ ]:
print("delete:", requests.delete(instance, headers=v2_headers()).status_code)
resp = requests.get(f"{instance}/subscription", headers=v2_headers())
print("after delete:", resp.status_code, resp.json())

## The same with the SDK

`Client.produce` and `Client.produce_batch` send records to the produce routes. A value of bytes is sent as `BINARY` and any other value, a `str` included, as `JSON`, which `json.loads` reads back the same as the SDK's `KafkaProducer` output; a `str` key is sent as `STRING`. `value_type` and `key_type` choose the type instead. `RESTConsumer` wraps the consumer routes. It creates its instance when it is constructed and deletes it at the end of the `with` block. A `binary` instance's keys and values come back as bytes, and header values are bytes in both formats. When the service answers 40403, `RESTConsumer` creates the instance again with the same name and settings, subscribes or assigns as before, and retries the request once. The new instance starts from the group's committed offsets, so records fetched since the last commit can come again. A `commit()` without offsets is not retried: when the records fetched since the last commit came from an instance that is gone, it sends nothing and raises `CommitLostError`.

In [ ]:
print(c.produce(topic, {"step": 6, "loss": 0.30}, key="job-42"))
batch = c.produce_batch(
    topic, [{"value": {"step": 7}}, {"value": "done"}, {"value": b"\x00\xff"}]
)
print(
    "successes:",
    [(s["id"], s["offset"]) for s in batch["successes"]],
    "failures:",
    batch["failures"],
)

with RESTConsumer(group, client=c, name="sdk-inst", format="binary") as consumer:
    consumer.subscribe([topic])
    records = []
    for _ in range(5):
        records += consumer.fetch(timeout_ms=5000)
        if records and records[-1]["offset"] >= 7:
            break
    for record in records:
        print(record["offset"], record["key"], record["value"])
    consumer.commit()
    print("committed:", consumer.committed([(topic, 0)]))

The group had committed offset 2 earlier, so this instance starts at offset 2.

## Clean up

Delete the topic this notebook created.

In [ ]:
print(json.dumps(c.delete_topic(topic), indent=2))